# 📡 Radar politiques publiques — STM (ADEME)

Ce notebook sert à **tester et régler** le radar (mots-clés, sources) sans rien installer.
La collecte « officielle » tourne toute seule chaque matin dans GitHub (action *Collecte radar*).

**Mode d'emploi** : exécutez les cellules une par une (▶️). La première collecte prend 3 à 6 minutes.

## 1. Récupérer le code\nSi le dépôt est privé, Colab demandera de se connecter à GitHub (ou rendez-le public le temps du test).

In [ ]:
DEPOT = "https://github.com/GabrielPlassat/radar_politique.git"
!rm -rf radar_politique && git clone -q $DEPOT
%cd radar_politique
!pip install -q -r requirements.txt

## 2. Vérifier que tout fonctionne (tests hors ligne, quelques secondes)

In [ ]:
!python -m pytest -q tests

## 3. Lancer une collecte
Toutes les sources, ou seulement certaines : `--sources an senat jorf consultations_mte commission_ue`.

In [ ]:
!python collecte.py

## 4. Bilan par source
Une ligne ⚠️ ERREUR = la source n'a pas répondu ; les autres sources sont quand même collectées.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
journal = pd.read_csv("data/journal_collecte.csv")
journal[journal["date"] == journal["date"].max()]

## 5. Ce que le radar a retenu

In [ ]:
textes = pd.read_csv("data/textes.csv", dtype=str).fillna("")
print(len(textes), "textes retenus")
print(textes["source"].value_counts().to_string())
textes[["source", "statut", "titre", "etape", "date_etape", "themes"]].head(40)

## 6. Régler les mots-clés
Textes retenus **par thème** : si un thème ramène trop de bruit, modifiez `config/themes.yaml` dans GitHub (mots, exclusions, score minimum), puis relancez les cellules 1 et 3.

In [ ]:
(textes.assign(theme=textes["themes"].str.split(" ; ")).explode("theme")
   .groupby("theme")["titre"].agg(["count", lambda s: " | ".join(s.head(3).str[:70])])
   .rename(columns={"count": "nb", "<lambda_0>": "exemples"}).sort_values("nb", ascending=False))

## 7. Acteurs déjà financés par l'ADEME (test)

In [ ]:
from radar.ademe import chercher_aides, synthese_beneficiaires
synthese_beneficiaires(chercher_aides(["propulsion vélique", "navire"])).head(15)